In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [3]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *

In [4]:
envconfig = EnvConfig()
cfg = envconfig

In [5]:
env = RideShareEnv(config=cfg)
G = env.G

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:424: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


In [6]:
s, info = env.reset()

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:424: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


In [7]:
pending_req = env.observation_curr["pending_requests"]

In [8]:
from waymo_agent.data_classes.requests import RequestDF


RequestDF.generate_empty(num_rows=5)

,request_id,request_dt,pickup_node_id,pickup_x_norm,pickup_y_norm,cust_id,cust_bias,cust_temperature,dropoff_node_id,dropoff_x_norm,...,route_nodes,curr_start_node,curr_end_node,route_dist_on_edge,distance_meters,est_cost,price,max_wait_time,wait_time,status
0,-1,2025-01-01,0,0.0,0.0,0,0.0,0.0,0,0.0,...,[],0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,-1,2025-01-01,0,0.0,0.0,0,0.0,0.0,0,0.0,...,[],0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,-1,2025-01-01,0,0.0,0.0,0,0.0,0.0,0,0.0,...,[],0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,-1,2025-01-01,0,0.0,0.0,0,0.0,0.0,0,0.0,...,[],0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,-1,2025-01-01,0,0.0,0.0,0,0.0,0.0,0,0.0,...,[],0,0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [9]:
out = ActiveRideDF.generate_empty(num_rows=env.num_vehicles)

In [10]:
type(out)

waymo_agent.data_classes.active_rides.ActiveRideDF

In [11]:
inspect_graph(env.G)

Graph has 782 nodes and 2746 edges.
Sample node -sample_node_id=42430646- attributes: {'y': 40.7576744, 'x': -74.0005846, 'highway': 'traffic_signals', 'street_count': 3, 'lambda': 0.0015, 'x_centered': -1771.2233, 'y_centered': 278.1597, 'x_norm': -0.2481, 'y_norm': 0.039}

Sample edge attributes: {'osmid': 1189567313, 'highway': 'primary', 'lanes': 1, 'maxspeed': 25.0, 'name': 'Grand Street', 'oneway': False, 'reversed': False, 'length': 292.72416271837477, 'geometry': <LINESTRING (-73.987 40.716, -73.987 40.716, -73.984 40.715)>, 'travel_time_minutes': 0.7025}

cls.edge_length_unit='meters', cls.speed_unit='km/h', cls.time_unit='minutes'


In [12]:
env.edge_df.sample(2)

,source,target,key,osmid,highway,lanes,maxspeed,name,oneway,ref,reversed,length,tunnel,geometry,travel_time_minutes
2294,4288677093,42447009,0,569345468,secondary,3,25.0,East 59th Street,False,NaN,True,153.800432,NaN,-1,0.3691
8,42421741,42421737,0,420625567,secondary,2,30.0,West 106th Street,False,NaN,False,271.350832,NaN,-1,0.5427


In [13]:
env.node_df.sample(2)

,node_id,y,x,street_count,lambda,x_centered,y_centered,x_norm,y_norm
295,42442947,40.751748,-73.970804,4,0.001,750.2708,-350.7029,0.1051,-0.0491
619,3786728763,40.801167,-73.929562,3,0.000,4164.9672,5176.9409,0.5834,0.7252


In [14]:
route_list = pending_req.route_nodes[0]
starts, ends = route_list[0:-1], route_list[1:]

In [15]:
from waymo_agent.osmnx.traverse_graph import (
    precompute_path_timesteps,
    f_edges_start_end_node,
    time_to_edge_progress,
    edge_progress_to_time,
    step_along_route,
)

route_enriched = precompute_path_timesteps(env, route_list)

In [16]:
route = precompute_path_timesteps(env, route_list)

In [17]:
route

,source,target,osmid,maxspeed,name,length,travel_time_minutes,CumTravelTime,DistanceRemaining,node_id_src,x_src,y_src,x_norm_src,y_norm_src,node_id_tgt,x_tgt,y_tgt,x_norm_tgt,y_norm_tgt
0,588455698,42438531,46154287,25.0,Montgomery Street,27.052699,0.0649,0.0649,4264.949073,588455698,-73.984656,40.710483,-0.0513,-0.6927,42438531,-73.984704,40.710724,-0.0519,-0.6890
1,42438531,370888100,46629376,28.0,South Street,383.505586,0.8218,0.8867,4237.896374,42438531,-73.984704,40.710724,-0.0519,-0.6890,370888100,-73.980169,40.710970,0.0017,-0.6845
2,370888100,42423296,958188024,40.0,FDR Drive,277.343928,0.4160,1.3027,3854.390788,370888100,-73.980169,40.710970,0.0017,-0.6845,42423296,-73.977759,40.712523,0.0300,-0.6601
3,42423296,42450820,958188024,40.0,FDR Drive,362.143076,0.5432,1.8459,3577.046860,42423296,-73.977759,40.712523,0.0300,-0.6601,42450820,-73.976182,40.715552,0.0481,-0.6127
4,42450820,42423039,958188025,40.0,FDR Drive,101.156719,0.1517,1.9976,3214.903784,42450820,-73.976182,40.715552,0.0481,-0.6127,42423039,-73.975741,40.716398,0.0531,-0.5995
5,42423039,42457401,32935473,32.0,FDR Drive,272.389320,0.5107,2.5083,3113.747065,42423039,-73.975741,40.716398,0.0531,-0.5995,42457401,-73.975113,40.718796,0.0601,-0.5621
6,42457401,42454391,822128686,25.0,FDR Drive,20.514659,0.0492,2.5575,2841.357745,42457401,-73.975113,40.718796,0.0601,-0.5621,42454391,-73.975082,40.718979,0.0605,-0.5593
7,42454391,42423549,46201667,28.0,FDR Drive,237.089538,0.5080,3.0655,2820.843086,42454391,-73.975082,40.718979,0.0605,-0.5593,42423549,-73.974715,40.721087,0.0644,-0.5265
8,42423549,370898427,432550168,40.0,FDR Drive,1034.834981,1.5523,4.6178,2583.753548,42423549,-73.974715,40.721087,0.0644,-0.5265,370898427,-73.972360,40.729881,0.0907,-0.3894
9,370898427,7147636389,32936385,40.0,NaN,180.485985,0.2707,4.8885,1548.918567,370898427,-73.972360,40.729881,0.0907,-0.3894,7147636389,-73.973937,40.730961,0.0718,-0.3728


In [18]:
pending_req.head(1)

,request_id,request_dt,pickup_node_id,pickup_x_norm,pickup_y_norm,cust_id,cust_bias,cust_temperature,dropoff_node_id,dropoff_x_norm,...,route_nodes,curr_start_node,curr_end_node,route_dist_on_edge,distance_meters,est_cost,price,max_wait_time,wait_time,status
0,5,2025-01-05 09:05:11.268574,588455698,-0.0513,-0.6927,49686,-0.731183,2.142336,42436586,-0.033,...,"[588455698, 42438531, 370888100, 42423296, 424...",588455698,42438531,0.0,4264.949073,4.264949,NaN,0.25,0.0,0


In [19]:
step_along_route(env, pending_req.head(1))

,request_id,request_dt,pickup_node_id,pickup_x_norm,pickup_y_norm,cust_id,cust_bias,cust_temperature,dropoff_node_id,dropoff_x_norm,...,curr_start_node,curr_end_node,route_dist_on_edge,distance_meters,est_cost,price,max_wait_time,wait_time,status,DistanceRemaining
0,5,2025-01-05 09:05:11.268574,588455698,-0.0513,-0.6927,49686,-0.731183,2.142336,42436586,-0.033,...,370888100,42423296,75.536219,4264.949073,4.264949,NaN,0.25,0.0,0,3854.390788


In [20]:
for row in pending_req.itertuples():
    # print(row)
    print(row.route_nodes)
    print(row.route_nodes[0])
    break

[588455698, 42438531, 370888100, 42423296, 42450820, 42423039, 42457401, 42454391, 42423549, 370898427, 7147636389, 5145531517, 5137978709, 42439191, 42448714, 42448707, 42442889, 42448701, 42436586]
588455698
